# CRAG Reproduction & Analysis — Complete Pipeline
**Corrective Retrieval-Augmented Generation** reproduced on PopQA with a local quantized Qwen2.5-7B backend.

This notebook runs the full study end to end:
1. Setup + local model + embeddings
2. Data (PopQA) + dev/eval splits
3. CRAG components (calibrated grader, rewriter, web search, generator, re-ranking)
4. The CRAG state machine (LangGraph)
5. Two corpora: **easy** (eval subjects only) and **hard** (eval + 300 distractors)
6. Experiments: Vanilla RAG vs CRAG on both corpora
7. Comparison analysis


## Section 0 — Installs


In [4]:
# Install everything (run once, then RESTART runtime)
!pip -q install "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langgraph>=0.2" \
  "langchain-huggingface" "faiss-cpu" "sentence-transformers" "datasets" "rank_bm25" "gradio"
!pip -q install bitsandbytes accelerate
!pip -q install ddgs hf_transfer
print("Installs done. Now: Runtime > Restart session, then run from Section 1.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.

## Section 1 — Login, warnings, imports, config

In [5]:
import os, warnings, logging
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

# Optional but recommended: add HF_TOKEN to Colab secrets for faster downloads
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("HF login OK")
except Exception as e:
    print("HF token not set (downloads still work, just slower)")

HF login OK


In [6]:
import json, re, time, random
import numpy as np
import torch, requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from ddgs import DDGS

EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE    = 256      # tokens (chunk-size sweep variable)
CHUNK_OVERLAP = 32
TOP_K         = 5
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## Section 2 — Local LLM backend (quantized Qwen2.5-7B on GPU)
First run downloads ~5.5 GB.

In [7]:
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # pre-quantized 4-bit
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"": 0})  # force GPU 0

gen_pipe = pipeline("text-generation", model=model, tokenizer=tok,
                    max_new_tokens=256, do_sample=False,        # greedy = deterministic
                    return_full_text=False, repetition_penalty=1.1)
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))

print("device:", next(model.parameters()).device,
      "| VRAM:", round(torch.cuda.memory_allocated()/1e9, 1), "GB")
print(llm.invoke("Reply with exactly: CRAG backend OK").content)

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

device: cuda:0 | VRAM: 5.5 GB
CRAG backend OK


In [8]:
# Embedding model for dense retrieval + re-ranking
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL,
                                   encode_kwargs={"normalize_embeddings": True})
print("Embeddings ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings ready.


## Section 3 — Data: PopQA, scorer, dev/eval splits

In [9]:
ds = load_dataset("akariasai/PopQA")
data = ds["test"]
print("PopQA loaded:", len(data), "questions")

def parse_gold_answers(ex):
    # possible_answers is a JSON-encoded STRING -> list
    return [a.strip() for a in json.loads(ex["possible_answers"]) if a and a.strip()]

def normalize(t): return t.lower().strip()

def exact_match(pred, gold):
    # PopQA accuracy: any gold answer is a substring of the prediction
    p = normalize(pred)
    return any(normalize(g) in p for g in gold)

random.seed(42)
dev_set = [data[i] for i in random.sample(range(len(data)), 50)]   # for tuning

random.seed(123)
dev_ids = {e["id"] for e in dev_set}
pool = [data[i] for i in range(len(data)) if data[i]["id"] not in dev_ids]
eval_set = random.sample(pool, 100)                                # held-out
print("dev:", len(dev_set), "| eval:", len(eval_set), "(disjoint)")

README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


test.tsv:   0%|          | 0.00/5.21M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14267 [00:00<?, ? examples/s]

PopQA loaded: 14267 questions
dev: 50 | eval: 100 (disjoint)


## Section 4 — Reusable corpus builders (fetch Wikipedia + chunk)

In [10]:
session = requests.Session()
session.headers.update({"User-Agent": "CRAG-course-project/1.0 (educational)"})
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1.5, status_forcelist=[429,500,502,503,504],
    respect_retry_after_header=True)))

def fetch_wikipedia_extract(title):
    r = session.get("https://en.wikipedia.org/w/api.php",
        params={"action":"query","prop":"extracts","explaintext":1,
                "titles":title,"format":"json","redirects":1}, timeout=30)
    r.raise_for_status()
    return next(iter(r.json()["query"]["pages"].values())).get("extract", "")

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(EMBED_MODEL),
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

def fetch_articles(titles, pause=0.5):
    arts = {}
    titles = sorted(set(titles))
    print(f"Fetching {len(titles)} articles...")
    for i, t in enumerate(titles, 1):
        try:
            txt = fetch_wikipedia_extract(t)
            if txt: arts[t] = txt
        except Exception:
            pass
        time.sleep(pause)
        if i % 50 == 0: print(f"  ...{i} done")
    print(f"Fetched {len(arts)}/{len(titles)}.")
    return arts

def chunk_articles(arts):
    docs = []
    for title, text in arts.items():
        for ch in splitter.split_text(text):
            docs.append(Document(page_content=ch,
                        metadata={"title": title, "source": "wikipedia"}))
    return docs

## Section 5 — CRAG components
Calibrated grader (loosened to avoid over-rejection), query rewriter, web search, generator, and the re-ranking helper.

In [11]:
# --- Retrieval evaluator (CALIBRATED: prefers Correct/Ambiguous when unsure) ---
GRADER_PROMPT = """You are a retrieval evaluator. Decide whether the retrieved documents
are likely to help answer the question about the entity named in it.

Question:
{question}

Retrieved documents:
{documents}

Guidance:
- Score "Correct" if the documents are about the right entity AND plausibly contain the answer.
- Score "Ambiguous" only if the documents are about the right entity but clearly lack the specific fact.
- Score "Incorrect" ONLY if the documents are about a clearly DIFFERENT entity/topic, or contain
  nothing related to the question. When unsure, prefer "Correct" or "Ambiguous" over "Incorrect".

Respond with ONLY a JSON object and nothing else:
{{"reasoning": "<one short sentence>", "score": "<Correct|Ambiguous|Incorrect>"}}"""

_VALID = {"Correct", "Ambiguous", "Incorrect"}

def _extract_json(t):
    m = re.search(r"\{.*\}", t, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def grade_retrieval(question, docs):
    dt = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))
    resp = llm.invoke(GRADER_PROMPT.format(question=question, documents=dt))
    p = _extract_json(resp.content)
    if p and p.get("score") in _VALID:
        return {"score": p["score"], "reasoning": p.get("reasoning", "")}
    low = resp.content.lower()
    for lab in ["incorrect", "ambiguous", "correct"]:
        if lab in low: return {"score": lab.capitalize(), "reasoning": "fallback"}
    return {"score": "Ambiguous", "reasoning": "default"}

In [12]:
# --- Query rewriter + web search (corrective source) ---
REWRITE_PROMPT = """You are rewriting a question into a web search query that will
find the answer. Keep ALL named entities exactly as written, and add a disambiguating
word for the entity type if implied (e.g. song, film, book, person, city).
Return ONLY the search query, nothing else.

Question: {question}
Search query:"""

def rewrite_query(question):
    return llm.invoke(REWRITE_PROMPT.format(question=question)).content.strip().strip('"')

def web_search(query, max_results=5):
    docs = []
    try:
        with DDGS() as d:
            for r in d.text(query, max_results=max_results):
                docs.append(Document(
                    page_content=f"{r.get('title','')}\n{r.get('body','')}",
                    metadata={"title": r.get("title",""), "source": "web",
                              "url": r.get("href","")}))
    except Exception as e:
        print("web_search error:", e)
    return docs

In [13]:
# --- Generator + re-ranking ---
GENERATOR_PROMPT = """Answer the question using ONLY the context below.
Be concise — give just the answer (a name, term, or short phrase), not a full sentence.
If the context does not contain the answer, reply exactly: I don't know.

Context:
{context}

Question: {question}
Answer:"""

def generate_answer(question, docs):
    ctx = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))
    return llm.invoke(GENERATOR_PROMPT.format(context=ctx, question=question)).content.strip()

def rerank_docs(question, docs, top_k=5):
    """Re-rank docs by embedding similarity to the question; keep top_k (cuts noise)."""
    if not docs: return docs
    qv = np.array(embeddings.embed_query(question))
    dv = np.array(embeddings.embed_documents([d.page_content for d in docs]))
    order = np.argsort(dv @ qv)[::-1][:top_k]
    return [docs[i] for i in order]

## Section 6 — CRAG state machine (LangGraph)
`retrieve -> grade -> {Correct: generate | else: websearch(augment + re-rank) -> generate}`.
`ACTIVE_RETRIEVER` is swappable so the same graph runs over either corpus.

In [14]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class CRAGState(TypedDict):
    question: str
    documents: List[Document]
    grade: str
    generation: str
    steps: List[str]

ACTIVE_RETRIEVER = None   # set to eval_dense or hard_dense before running

def node_retrieve(s):
    return {"documents": ACTIVE_RETRIEVER.invoke(s["question"]),
            "steps": s.get("steps", []) + ["retrieve"]}

def node_grade(s):
    r = grade_retrieval(s["question"], s["documents"])
    return {"grade": r["score"], "steps": s["steps"] + [f"grade={r['score']}"]}

def node_websearch(s):
    web = web_search(rewrite_query(s["question"]))
    combined = s["documents"] + web                       # augment (keep local)
    docs = rerank_docs(s["question"], combined, top_k=TOP_K)   # trim noise
    lab = "websearch(augment)" if s["grade"] == "Ambiguous" else "websearch(incorrect+augment)"
    return {"documents": docs, "steps": s["steps"] + [lab + "+rerank"]}

def node_generate(s):
    return {"generation": generate_answer(s["question"], s["documents"]),
            "steps": s["steps"] + ["generate"]}

def route_after_grade(s):
    return "generate" if s["grade"] == "Correct" else "websearch"

def build_crag_app():
    b = StateGraph(CRAGState)
    b.add_node("retrieve", node_retrieve)
    b.add_node("grade", node_grade)
    b.add_node("websearch", node_websearch)
    b.add_node("generate", node_generate)
    b.add_edge(START, "retrieve")
    b.add_edge("retrieve", "grade")
    b.add_conditional_edges("grade", route_after_grade,
                            {"generate": "generate", "websearch": "websearch"})
    b.add_edge("websearch", "generate")
    b.add_edge("generate", END)
    return b.compile()

print("Graph functions defined.")

Graph functions defined.


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Section 7 — Build the two corpora
**Easy** = 100 eval subjects. **Hard** = eval subjects + 300 distractor articles (~2-3 min fetch + embed).

In [15]:
# Easy corpus: just the eval subjects
eval_articles = fetch_articles([e["s_wiki_title"] for e in eval_set])
eval_docs = chunk_articles(eval_articles)
eval_vs = FAISS.from_documents(eval_docs, embeddings)
eval_dense = eval_vs.as_retriever(search_kwargs={"k": TOP_K})
print("EASY corpus:", len(eval_docs), "chunks from", len(eval_articles), "articles")

Fetching 100 articles...
  ...50 done
  ...100 done
Fetched 100/100.
EASY corpus: 1747 chunks from 100 articles


In [16]:
# Hard corpus: eval subjects + 300 distractor articles
random.seed(777)
used = {e["id"] for e in eval_set} | {e["id"] for e in dev_set}
dpool = [data[i] for i in range(len(data)) if data[i]["id"] not in used]
dexs = random.sample(dpool, 300)
distractor_articles = fetch_articles([e["s_wiki_title"] for e in dexs], pause=0.4)

hard_docs = eval_docs + chunk_articles(distractor_articles)
print("Embedding hard corpus (~1-2 min)...")
hard_vs = FAISS.from_documents(hard_docs, embeddings)
hard_dense = hard_vs.as_retriever(search_kwargs={"k": TOP_K})
print("HARD corpus:", len(hard_docs), "chunks from",
      len(eval_articles) + len(distractor_articles), "articles")

Fetching 298 articles...
  ...50 done
  ...100 done
  ...150 done
  ...200 done
  ...250 done
Fetched 297/298.
Embedding hard corpus (~1-2 min)...
HARD corpus: 6262 chunks from 397 articles


## Section 8 — Evaluation harness (Vanilla RAG vs CRAG)
Reusable; saves per-run results to disk and prints progress.

In [17]:
def run_eval(retriever, label, path):
    """Run vanilla RAG and CRAG over eval_set using `retriever`. Saves to `path`."""
    global ACTIVE_RETRIEVER, crag_app
    ACTIVE_RETRIEVER = retriever
    crag_app = build_crag_app()                # graph uses this retriever

    def vanilla(q):
        return generate_answer(q, retriever.invoke(q))

    res, t0 = {}, time.time()
    for i, ex in enumerate(eval_set, 1):
        q, gold = ex["question"], parse_gold_answers(ex)
        ba = vanilla(q)
        co = crag_app.invoke({"question": q, "steps": []})
        res[str(ex["id"])] = {
            "question": q, "gold": gold,
            "baseline_ans": ba, "baseline_correct": exact_match(ba, gold),
            "crag_ans": co["generation"], "crag_correct": exact_match(co["generation"], gold),
            "path": co["steps"],
        }
        if i % 10 == 0:
            with open(path, "w") as f: json.dump(res, f)
            print(f"  [{label}] {i}/{len(eval_set)} | {time.time()-t0:.0f}s")
    with open(path, "w") as f: json.dump(res, f)
    print(f"[{label}] DONE -> {path}")
    return res

## Section 9 — Run the experiments
Each run is ~10-30 min on a T4 (CRAG makes several model calls per question).

In [18]:
# Experiment A: calibrated CRAG vs vanilla, EASY corpus
results_easy = run_eval(eval_dense, "calibrated/easy", "/content/results_easy.json")

  [calibrated/easy] 10/100 | 249s
  [calibrated/easy] 20/100 | 522s
  [calibrated/easy] 30/100 | 790s
  [calibrated/easy] 40/100 | 1047s
  [calibrated/easy] 50/100 | 1309s
  [calibrated/easy] 60/100 | 1576s
  [calibrated/easy] 70/100 | 1849s
  [calibrated/easy] 80/100 | 2100s
  [calibrated/easy] 90/100 | 2356s
  [calibrated/easy] 100/100 | 2622s
[calibrated/easy] DONE -> /content/results_easy.json


In [19]:
# Experiment B: calibrated CRAG + re-rank vs vanilla, HARD corpus
results_hard = run_eval(hard_dense, "calibrated+rerank/hard", "/content/results_hard.json")

  [calibrated+rerank/hard] 10/100 | 238s
  [calibrated+rerank/hard] 20/100 | 507s
  [calibrated+rerank/hard] 30/100 | 797s
  [calibrated+rerank/hard] 40/100 | 1069s
  [calibrated+rerank/hard] 50/100 | 1345s
  [calibrated+rerank/hard] 60/100 | 1604s
  [calibrated+rerank/hard] 70/100 | 1861s
  [calibrated+rerank/hard] 80/100 | 2126s
  [calibrated+rerank/hard] 90/100 | 2381s
  [calibrated+rerank/hard] 100/100 | 2621s
[calibrated+rerank/hard] DONE -> /content/results_hard.json


## Section 10 — Comparison analysis
Accuracy, crossover (CRAG fixed vs broke), and path distribution for each config.

In [20]:
from collections import Counter

def path_type(steps):
    s = " ".join(steps)
    if "websearch(replace)" in s:  return "Incorrect->web(replace)"
    if "incorrect+augment" in s:   return "Incorrect->web(augment)"
    if "websearch(augment)" in s:  return "Ambiguous->web(augment)"
    return "Correct->generate"

def analyze(res, label):
    n = len(res)
    bc = sum(r["baseline_correct"] for r in res.values())
    cc = sum(r["crag_correct"] for r in res.values())
    fixed = sum(1 for r in res.values() if r["crag_correct"] and not r["baseline_correct"])
    broke = sum(1 for r in res.values() if r["baseline_correct"] and not r["crag_correct"])
    print("=" * 60)
    print(f"  {label}  (n={n})")
    print("=" * 60)
    print(f"  Vanilla RAG : {bc/n:.1%}  ({bc}/{n})")
    print(f"  CRAG        : {cc/n:.1%}  ({cc}/{n})")
    print(f"  Delta       : {(cc-bc)/n:+.1%}")
    print(f"  CRAG fixed {fixed} | CRAG broke {broke}")
    print("  Path distribution:")
    paths = Counter(path_type(r["path"]) for r in res.values())
    for p, c in paths.most_common():
        sub = [r for r in res.values() if path_type(r["path"]) == p]
        acc = sum(r["crag_correct"] for r in sub) / len(sub)
        print(f"    {p:28s}: {c:3d} q, {acc:.0%} correct")
    print()
    return {"label": label, "vanilla": bc/n, "crag": cc/n, "delta": (cc-bc)/n}

s1 = analyze(results_easy, "Calibrated grader (EASY corpus)")
s2 = analyze(results_hard, "Calibrated + re-rank (HARD corpus)")

print("=" * 60); print("  SUMMARY"); print("=" * 60)
print(f"  {'Config':<42}{'Vanilla':>9}{'CRAG':>8}{'Delta':>8}")
for s in [s1, s2]:
    print(f"  {s['label']:<42}{s['vanilla']:>8.0%}{s['crag']:>8.0%}{s['delta']:>+8.0%}")

  Calibrated grader (EASY corpus)  (n=100)
  Vanilla RAG : 61.0%  (61/100)
  CRAG        : 65.0%  (65/100)
  Delta       : +4.0%
  CRAG fixed 4 | CRAG broke 0
  Path distribution:
    Correct->generate           :  80 q, 71% correct
    Incorrect->web(augment)     :  15 q, 40% correct
    Ambiguous->web(augment)     :   5 q, 40% correct

  Calibrated + re-rank (HARD corpus)  (n=100)
  Vanilla RAG : 65.0%  (65/100)
  CRAG        : 62.0%  (62/100)
  Delta       : -3.0%
  CRAG fixed 0 | CRAG broke 3
  Path distribution:
    Correct->generate           :  77 q, 74% correct
    Incorrect->web(augment)     :  18 q, 22% correct
    Ambiguous->web(augment)     :   5 q, 20% correct

  SUMMARY
  Config                                      Vanilla    CRAG   Delta
  Calibrated grader (EASY corpus)                61%     65%     +4%
  Calibrated + re-rank (HARD corpus)             65%     62%     -3%


## Section 11 — Error taxonomy (qualitative analysis)
Inspect the cases where CRAG and the baseline disagree, to categorize failure types.

In [21]:
# Show disagreement cases for manual taxonomy labelling
def show_disagreements(res, label, limit=15):
    print(f"\n### {label} — CRAG vs baseline disagreements ###")
    shown = 0
    for r in res.values():
        if r["baseline_correct"] != r["crag_correct"]:
            tag = "CRAG FIXED" if r["crag_correct"] else "CRAG BROKE"
            print(f"\n[{tag}] Q: {r['question']}")
            print(f"   gold    : {r['gold']}")
            print(f"   baseline: {r['baseline_ans'][:60]!r}")
            print(f"   crag    : {r['crag_ans'][:60]!r}")
            print(f"   path    : {' -> '.join(r['path'])}")
            shown += 1
            if shown >= limit: break

show_disagreements(results_hard, "HARD corpus")


### HARD corpus — CRAG vs baseline disagreements ###

[CRAG BROKE] Q: Who is the author of Mars?
   gold    : ['Marc Hempel']
   baseline: 'Mark Wheatley and Marc Hempel'
   crag    : "I don't know."
   path    : retrieve -> grade=Incorrect -> websearch(incorrect+augment)+rerank -> generate

[CRAG BROKE] Q: Who was the director of Live and Learn?
   gold    : ['Carl Franklin', 'Carl Michael Franklin']
   baseline: 'Carl Franklin'
   crag    : 'Jeff Fowler'
   path    : retrieve -> grade=Incorrect -> websearch(incorrect+augment)+rerank -> generate

[CRAG BROKE] Q: Who was the producer of The Walk?
   gold    : ['Robert Zemeckis', 'Robert L. Zemeckis', 'Robert Lee Zemeckis']
   baseline: 'Robert Zemeckis'
   crag    : "I don't know."
   path    : retrieve -> grade=Incorrect -> websearch(incorrect+augment)+rerank -> generate


In [24]:
# STEP 11b (50-question version): eval harness over a chosen subset
def run_eval_subset(retriever, label, path, questions):
    global ACTIVE_RETRIEVER, crag_app
    ACTIVE_RETRIEVER = retriever
    crag_app = build_crag_app()
    def vanilla(q): return generate_answer(q, retriever.invoke(q))
    res, t0 = {}, time.time()
    for i, ex in enumerate(questions, 1):
        q, gold = ex["question"], parse_gold_answers(ex)
        ba = vanilla(q)
        co = crag_app.invoke({"question": q, "steps": []})
        res[str(ex["id"])] = {
            "question": q, "gold": gold,
            "baseline_ans": ba, "baseline_correct": exact_match(ba, gold),
            "crag_ans": co["generation"], "crag_correct": exact_match(co["generation"], gold),
            "path": co["steps"],
        }
        if i % 10 == 0:
            with open(path, "w") as f: json.dump(res, f)
            print(f"  [{label}] {i}/{len(questions)} | {time.time()-t0:.0f}s")
    with open(path, "w") as f: json.dump(res, f)
    print(f"[{label}] DONE")
    return res

In [22]:
# STEP 11a: Build BM25, dense, hybrid on the EASY corpus
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

eval_bm25 = BM25Retriever.from_documents(eval_docs)
eval_bm25.k = TOP_K
# eval_dense already exists from Section 7
eval_hybrid = EnsembleRetriever(retrievers=[eval_bm25, eval_dense], weights=[0.5, 0.5])
print("Three easy-corpus retrievers ready: bm25, dense, hybrid")

Three easy-corpus retrievers ready: bm25, dense, hybrid


In [25]:
# STEP 11b: Evaluate vanilla vs CRAG under each retriever (easy corpus)
eval50 = eval_set[:50]
retriever_results = {}
for name, retr in [("bm25", eval_bm25), ("dense", eval_dense), ("hybrid", eval_hybrid)]:
    print(f"\n===== Retriever: {name} =====")
    retriever_results[name] = run_eval_subset(retr, f"retr={name}", f"/content/results_retr_{name}.json", eval50)


===== Retriever: bm25 =====
  [retr=bm25] 10/50 | 266s
  [retr=bm25] 20/50 | 522s
  [retr=bm25] 30/50 | 791s
  [retr=bm25] 40/50 | 1049s
  [retr=bm25] 50/50 | 1312s
[retr=bm25] DONE

===== Retriever: dense =====
  [retr=dense] 10/50 | 249s
  [retr=dense] 20/50 | 519s
  [retr=dense] 30/50 | 788s
  [retr=dense] 40/50 | 1045s
  [retr=dense] 50/50 | 1307s
[retr=dense] DONE

===== Retriever: hybrid =====
  [retr=hybrid] 10/50 | 419s
  [retr=hybrid] 20/50 | 862s
  [retr=hybrid] 30/50 | 1316s
  [retr=hybrid] 40/50 | 1746s
  [retr=hybrid] 50/50 | 2165s
[retr=hybrid] DONE


In [26]:
# STEP 11c: Comparison table
def summarize(res):
    n = len(res)
    return (sum(r["baseline_correct"] for r in res.values())/n,
            sum(r["crag_correct"] for r in res.values())/n)

print(f"{'Retriever':<10}{'Vanilla':>9}{'CRAG':>8}{'Δ(CRAG-van)':>14}")
for name in ["bm25", "dense", "hybrid"]:
    v, c = summarize(retriever_results[name])
    print(f"{name:<10}{v:>8.0%}{c:>8.0%}{c-v:>+13.0%}")

Retriever   Vanilla    CRAG   Δ(CRAG-van)
bm25           20%     30%         +10%
dense          60%     64%          +4%
hybrid         64%     66%          +2%
